In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 56.5 MB/s eta 0:00:00


In [2]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Allocated VRAM: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB")

CUDA Available: True
Device: Tesla T4
Allocated VRAM: 0.00 GB


In [3]:
import json
import urllib.request

# Download training dataset directly into Colab
raw_url = "https://raw.githubusercontent.com/Umanagalla27/p3-finetuning-mlops/main/data/processed/train.json"
urllib.request.urlretrieve(raw_url, "train.json")

with open("train.json") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} training samples. Sample 1:")
print(json.dumps(train_data[0], indent=2))

Loaded 1020 training samples. Sample 1:
{
  "instruction": "Extract the structured IT ticket metadata and output ONLY a valid JSON object matching the enterprise schema.",
  "input": "Monthly cloud invoice discrepancy of $1303 on account ACC-7337.",
  "output": "{\"ticket_id\": \"TKT-10724\", \"category\": \"billing\", \"urgency\": \"low\", \"recommended_tool\": \"audit_billing\", \"requires_human_approval\": false}"
}


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit NormalFloat Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for LoRA gradient checkpointing
model = prepare_model_for_kbit_training(model)
print(f"Model loaded in 4-bit! VRAM used: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded in 4-bit! VRAM used: 1.51 GB


In [5]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,                         # Rank: controls capacity of the adaptation
    lora_alpha=32,                # Scaling factor (standard ratio: alpha = 2 * r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [10]:
from datasets import Dataset

def format_and_tokenize(sample):
    text = (
        f"<|im_start|>system\n{sample['instruction']}<|im_end|>\n"
        f"<|im_start|>user\n{sample['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{sample['output']}<|im_end|>"
    )
    tokens = tokenizer(text, truncation=True, max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()  # Standard causal LM autoregressive loss
    return tokens

dataset = Dataset.from_list(train_data).map(format_and_tokenize, remove_columns=list(train_data[0].keys()))
print("Dataset ready! Sample input_ids length:", len(dataset[0]["input_ids"]))

Map:   0%|          | 0/1020 [00:00<?, ? examples/s]

Dataset ready! Sample input_ids length: 93


In [11]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./qwen-qlora-it-triage",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    learning_rate=2e-4,             # Standard QLoRA learning rate
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=20,
    fp16=True,                      # Fast mixed-precision on T4 GPU
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")
)

print("Starting QLoRA training on T4 GPU...")
trainer.train()

Starting QLoRA training on T4 GPU...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.135577
40,0.233010
60,0.183945
80,0.185845
100,0.178280
120,0.176285


TrainOutput(global_step=128, training_loss=0.4940292751416564, metrics={'train_runtime': 352.6831, 'train_samples_per_second': 5.784, 'train_steps_per_second': 0.363, 'total_flos': 1639214311833600.0, 'train_loss': 0.4940292751416564, 'epoch': 2.0})

In [12]:
adapter_dir = "./final_qlora_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA adapter saved successfully to {adapter_dir}!")

# Zip adapter for downloading (size is only ~35MB, not 3GB!)
!zip -r qlora_adapter.zip final_qlora_adapter

LoRA adapter saved successfully to ./final_qlora_adapter!
  adding: final_qlora_adapter/ (stored 0%)
  adding: final_qlora_adapter/adapter_config.json (deflated 60%)
  adding: final_qlora_adapter/README.md (deflated 65%)
  adding: final_qlora_adapter/adapter_model.safetensors (deflated 8%)
  adding: final_qlora_adapter/tokenizer_config.json (deflated 60%)
  adding: final_qlora_adapter/tokenizer.json (deflated 81%)
  adding: final_qlora_adapter/chat_template.jinja (deflated 71%)
